# Stage 4 — Check whether our training data is trustworthy

**What:** Open the labeled file, check its fingerprint and structure, then describe the labels, missing values, groups, and time coverage.

**Why:** A model can run on bad data and still produce a number. Data checks help us catch mistakes before training.

Your Stage 3 result had 10,110 rows: 7,228 label `0` and 2,882 label `1`. This stage verifies that saved result and checks its limits.

In [ ]:
# Cell 2 — Why: read the exact labeled file and verify its fingerprint.
from google.colab import drive
from pathlib import Path
import hashlib, json, pandas as pd

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/nyc311_mlops")
label_id = "20260926T110908975801Z"
label_dir = project / "labeled" / label_id
meta = json.loads((label_dir / "manifest.json").read_text())
saved = (label_dir / "training_rows.jsonl").read_bytes()
if hashlib.sha256(saved).hexdigest() != meta["sha256"]:
    raise ValueError("Labeled data changed")
df = pd.DataFrame([json.loads(line) for line in saved.splitlines()])
if len(df) != meta["rows"]:
    raise ValueError("Labeled row count changed")
print("Verified labeled rows:", len(df))
print("Raw snapshot used:", meta["raw_snapshot_id"])

In [ ]:
# Cell 3 — Why: stop if required columns, IDs, or labels are wrong.
required = {"unique_key", "created_date", "complaint_type", "borough", "label"}
if not required.issubset(df.columns):
    raise ValueError("Required columns are missing")
if df.empty or df["unique_key"].isna().any() or df["unique_key"].duplicated().any():
    raise ValueError("Empty data or invalid request IDs")
if not df["label"].isin([0, 1]).all():
    raise ValueError("Labels must be 0 or 1")
if "closed_date" in df.columns:
    raise ValueError("The future answer leaked into training inputs")
print("Data contract checks passed")

In [ ]:
# Cell 4 — Why: see blanks and the share of each answer.
display(df.isna().sum().to_frame("blank_values"))
counts = df["label"].value_counts().sort_index()
share = (counts / len(df) * 100).round(1)
display(pd.DataFrame({"requests": counts, "percent": share}))

In [ ]:
# Cell 5 — Why: see whether the late rate differs across groups.
by_borough = pd.crosstab(
    df["borough"].fillna("MISSING"), df["label"], normalize="index"
).round(3)
display(df["borough"].fillna("MISSING").value_counts().to_frame("requests"))
display(by_borough)

top_types = df["complaint_type"].value_counts().head(5).index
subset = df[df["complaint_type"].isin(top_types)]
display(subset["complaint_type"].value_counts().to_frame("requests"))
display(pd.crosstab(subset["complaint_type"], subset["label"], normalize="index").round(3))

In [ ]:
# Cell 6 — Why: one day's data cannot test how a model works later.
created = pd.to_datetime(df["created_date"], errors="coerce")
if created.isna().any():
    raise ValueError("Some creation times cannot be read")
days = created.dt.date.nunique()
print("First request:", created.min())
print("Last request:", created.max())
print("Distinct calendar days:", days)
if days == 1:
    print("We need more dates before a meaningful future-time test.")

# Stage 4 takeaway and quiz

**Why this stage mattered:** We verified the saved file, enforced a basic data contract, inspected label balance and groups, and found the time coverage. We will collect more historical dates before training and testing a model.

**Easy quiz:**

1. Why do we check the file's checksum before using it?
2. What problem would duplicate request IDs cause?
3. What do labels `0` and `1` mean?
4. Why is one day of data too narrow for a future-time test?

Send me the output of Cells 3, 4, and 6, plus your answers. Then we will broaden the data and build the first baseline model.